In [1]:
%pip install transformers accelerate datasets peft


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#   후보 1: 디자인 전문가 페르소나
#   > "당신은 자동차 디자인 트렌드와 역사에 정통한 '자동차 디자인 전문 AI'입니다. 특히 현대자동차의 디자인 철학인
#   '센슈어스 스포티니스'와 '플루이딕 스컬프처'를 깊이 이해하고 있습니다. 사용자의 질문에 대해, 전문 지식을 바탕으로
#   시각적이고 창의적인 관점에서 상세하게 설명해주세요."

#   후보 2: 디자인 컨설턴트 페르소나
#   > "당신은 새로운 자동차 디자인 프로토타입을 기획하는 '디자인 컨설턴트'입니다. 현대차뿐만 아니라 글로벌 자동차 디자인
#   트렌드를 폭넓게 이해하고 있으며, 이를 바탕으로 사용자가 디자인 영감을 얻을 수 있도록 돕습니다. 기술적, 미학적 관점을
#   통합하여 창의적인 아이디어를 제공하듯 답변해주세요."

#   후보 3: VQA (Visual Question Answering) 어시스턴트 페르소나
#   > "당신은 텍스트 설명을 바탕으로 자동차의 이미지를 상상하고, 디자인 컨셉을 구체화하는 '디자인 시각화 AI'입니다.
#   사용자의 질문에 대해, 마치 눈앞에 자동차가 있는 것처럼 형태, 라인, 재질, 색상 등을 풍부하고 생생하게 묘사하며
#   답변해주세요."

In [2]:
# pip install -U "transformers>=4.45.0" accelerate peft datasets

import os, json, torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model

# ---------------------------------------------
# 환경 설정 (권장)
# ---------------------------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cuda.matmul.allow_tf32 = True   # A100에서 속도/안정성 약간 도움

# ---------------------------------------------
# 0) 기본 설정
# ---------------------------------------------
SYSTEM_PROMPT = (
    "당신은 자동차 디자인 트렌드와 역사에 정통한 '자동차 디자인 전문 AI'입니다. "
    "특히 현대자동차의 디자인 철학인 '센슈어스 스포티니스'와 '플루이딕 스컬프처'를 깊이 이해하고 있습니다. "
    "사용자의 질문에 대해, 전문 지식을 바탕으로 시각적이고 창의적인 관점에서 상세하게 설명해주세요."
)

TRAIN_JSON_PATH = "./train.jsonl"
# VALID_JSON_PATH = "./validation.jsonl"  # 평가 비활성화면 생략 가능

MODEL_PATH = "./kanana1_5_8b_instruct_2505"  # 로컬 경로 또는 HF 모델 ID
OUTPUT_DIR = "./kanana_finetuned_model"

# 컷오프 비활성화
MAX_LEN = None

# ---------------------------------------------
# 1) 데이터 로드/포맷
# ---------------------------------------------
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            return json.load(f)  # JSON 배열
        except json.JSONDecodeError:
            f.seek(0)
            return [json.loads(line) for line in f]  # JSONL

def format_data_for_finetuning(raw_data):
    out = []
    for item in raw_data:
        msgs = item.get("messages", [])
        context = item.get("context")
        if not msgs:
            continue
        full = [{"role": "system", "content": SYSTEM_PROMPT}]
        if context:
            full.append({"role": "system", "content": f"다음은 참고 문맥입니다:\n{context}"})
        full += msgs
        out.append({"messages": full})
    return out

train_raw = load_data(TRAIN_JSON_PATH)
train_ds  = Dataset.from_list(format_data_for_finetuning(train_raw))

# (평가 비활성화라면 validation은 생략)
# valid_raw = load_data(VALID_JSON_PATH)
# valid_ds  = Dataset.from_list(format_data_for_finetuning(valid_raw))

# ---------------------------------------------
# 2) 모델/토크나이저 (A100 80GB: bf16 + LoRA)
# ---------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True, use_fast=True)
# config에 pad_token_id가 있으면 그대로 사용 (예: 128001). 없을 때만 eos로 대체.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # A100 bf16 권장
    device_map="auto",
)

# gradient_checkpointing과 궁합: 캐시 비활성화
if getattr(model.config, "use_cache", None):
    model.config.use_cache = False

# ---------------------------------------------
# 3) LoRA 구성 (Llama 계열 안전 타깃)
# ---------------------------------------------
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# gradient checkpointing + 입력 grad 요구 플래그 명시 (그래프 끊김 방지)
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ---------------------------------------------
# 4) 토큰화 (chat 템플릿 + pad 라벨 -100, 컷오프 없음)
#    - 초기엔 리스트 상태로 토큰화 → pad에서 한 번에 텐서화
# ---------------------------------------------
def to_prompt(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def tokenize_batch(examples):
    prompts = [to_prompt(m) for m in examples["messages"]]

    # (1) 가변 길이 토큰화 (텐서 변환 X)
    enc = tokenizer(
        prompts,
        add_special_tokens=True,
        padding=False,
        truncation=False,    # 컷오프 비활성화
        # return_tensors 생략
    )

    # (2) 배치 내 최장 길이에 맞춰 패딩 + 텐서화
    enc = tokenizer.pad(enc, padding=True, return_tensors="pt", pad_to_multiple_of=8)

    # (3) pad 라벨 -100 마스킹
    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100

    return {
        "input_ids": enc["input_ids"].tolist(),
        "attention_mask": enc["attention_mask"].tolist(),
        "labels": labels.tolist(),
    }

tokenized_train_dataset = train_ds.map(tokenize_batch, batched=True, remove_columns=["messages"])
# tokenized_valid_dataset = valid_ds.map(tokenize_batch, batched=True, remove_columns=["messages"])

# ---------------------------------------------
# 5) 훈련 설정 (체크포인트 저장 없음, 컷오프 없음)
#    - 80GB면 보통 batch 2 가능. 길이가 매우 길면 1로 낮추세요.
#    - transformers 버전에 따라 evaluation_strategy 인자가 없을 수 있으니 빼둠.
# ---------------------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                 # 필요시 5로 올리기
    per_device_train_batch_size=2,      # 80GB면 2도 여유. 길면 1로 낮추기
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,      # 유효 배치 16
    learning_rate=2e-4,
    weight_decay=0.01,
    bf16=True,
    gradient_checkpointing=True,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    save_strategy="no",                 # ★ 중간 체크포인트 저장 안 함
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
    optim="adamw_torch",
)

# ---------------------------------------------
# 6) Trainer 구성 (배치/타입 안정화를 위해 collator 지정)
# ---------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=None,                  # 평가 비활성화
    tokenizer=tokenizer,
    data_collator=default_data_collator # ✅ (batch, seq_len) 2D 보장 & dtype 정렬
)

print("Starting LoRA fine-tuning on A100 80GB (NO checkpoint, NO cutoff)...")
trainer.train()
print("✅ Fine-tuning complete.")

# ---------------------------------------------
# 7) 최종 저장 (LoRA 어댑터 + 토크나이저)
# ---------------------------------------------
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"📦 Saved LoRA adapter & tokenizer to {OUTPUT_DIR}")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 8,051,257,344 || trainable%: 0.2605


Map:   0%|          | 0/266 [00:00<?, ? examples/s]

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/tmp/ipykernel_589/1504757385.py:171: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting LoRA fine-tuning on A100 80GB (NO checkpoint, NO cutoff)...


Step,Training Loss
10,1.272200
20,1.035700
30,0.986800
40,0.953400
50,0.964400


✅ Fine-tuning complete.
📦 Saved LoRA adapter & tokenizer to ./kanana_finetuned_model
